In [1]:
import os
from dotenv import load_dotenv
from pathlib import Path
import numpy as np
import random

load_dotenv()
ROOT = Path(os.environ["DATA_ROOT"])
HF_HOME = ROOT / ".hf_cache"
os.environ["HF_HOME"] = str(HF_HOME)

import torch
from transformers import AutoTokenizer
from datasets import load_dataset

In [2]:
# Config
config = {
    'num_labels': 188,
    'seed': 42,
    'model_name': 'skt/A.X-Encoder-base',
}

In [3]:
random.seed(config['seed'])
np.random.seed(config['seed'])
torch.manual_seed(config['seed'])
torch.cuda.manual_seed_all(config['seed'])

## 데이터셋 확인

In [4]:
dataset = load_dataset("ingyoun/patent-clean-text")
dataset

data/train-00000-of-00002.parquet:   0%|          | 0.00/129M [00:00<?, ?B/s]

c:\workspace\patent_disc\.venv\Lib\site-packages\huggingface_hub\file_download.py:139: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\workspace\patent_disc\.hf_cache\hub\datasets--ingyoun--patent-clean-text. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


data/train-00001-of-00002.parquet:   0%|          | 0.00/121M [00:00<?, ?B/s]

data/test-00000-of-00001.parquet:   0%|          | 0.00/14.5M [00:00<?, ?B/s]

data/val-00000-of-00001.parquet:   0%|          | 0.00/14.2M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/201895 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/11271 [00:00<?, ? examples/s]

Generating val split:   0%|          | 0/11162 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['document_id', 'invention_title', 'abstract', 'claims', 'ipc_main', 'mno', 'label_ids', 'lno', 'kobert_len', 'length_bin'],
        num_rows: 201895
    })
    test: Dataset({
        features: ['document_id', 'invention_title', 'abstract', 'claims', 'ipc_main', 'mno', 'label_ids', 'lno', 'kobert_len', 'length_bin'],
        num_rows: 11271
    })
    val: Dataset({
        features: ['document_id', 'invention_title', 'abstract', 'claims', 'ipc_main', 'mno', 'label_ids', 'lno', 'kobert_len', 'length_bin'],
        num_rows: 11162
    })
})

## 토크나이저

In [ ]:
REV = "9708f9c404ace91efd25c06fac2d73413616f4ef"
tokenizer = AutoTokenizer.from_pretrained(config["model_name"], revision=REV)

In [6]:
print(f"Vocab Size: {tokenizer.vocab_size}")
print(f"Max Length: {tokenizer.model_max_length}")
print(f"Max Length: {tokenizer.all_special_tokens}")

Vocab Size: 49999
Max Length: 16384
Max Length: ['<s>', '<\\s>', '<unk>', '<sep>', '<pad>', '<cls>', '<mask>']


In [7]:
for k, v in dataset["train"][0].items():
    print(f"{k}: {v}")

document_id: kr20010002596b1
invention_title: 반도체 디바이스 시험방법 및 그의 장치
abstract: 피시험 반도체 디바이스로부터 판독되는 각 데이터와, 이들의 데이터에 동기하여 출력되는 기준 클록을 각각 약간씩 위상차가 부여된 다상 펄스의 스트로브 펄스로 샘플링하고, 이들 샘플링 출력으로부터 각 출력데이터의 변화점 위상과 기준 클록의 변화점 위상을 구하고, 이들 양자의 위상차를 각각 계측하고, 이 위상차가 미리 정한 조건의 범위내에 있는가 여부에 의하여 피시험 반도체 디바이스의 양부를 판정한다.
claims: 피시험 디바이스로부터 출력되는 기준클록의 변화점의 초기 위상위치로부터의 위상을 측정하고, 이 기준클록의 위상으로부터 이 기준클록에 동기하여 출력되는 각 데이터의 변화점까지의 위상차를 구하여, 이 위상차의 장단에 의하여 상기 피시험 디바이스의 그레이드를 평가하는 반도체 디바이스 시험방법에 있어서,각 테스트 사이클마다 초기 위상위치로부터 순차적으로 약간씩 위상차가 부여된 다상펄스를 발생시키는 단계,이 다상펄스를 스트로브펄스로 하여 각 테스트 사이클마다 상기 기준클록을 다상으로 샘플링하는 단계,이들 다상 샘플링 출력의 인접하는 출력의 비교로부터 상기 기준클록의 변화점을 검출하고, 이러한 변화점을 검출한 다상펄스의 상번호로부터 상기 기준클록의 변화점의 위상을 결정하는 단계를 포함하는 것을 특징으로 하는 반도체 디바이스 시험방법.피시험 디바이스로부터 출력되는 기준클록의 변화점의 초기 위상위치로부터의 위상을 측정하고, 이 기준클록의 위상으로부터 이 기준클록에 동기하여 출력되는 각 데이터의 변화점까지의 위상차를 구하고, 이 위상차의 장단에 의하여 상기 피시험 디바이스의 그레이드를 평가하는 반도체 디바이스 시험방법에 있어서,각 테스트 사이클마다, 출력되는 상기 기준클록의 변화점의 위상을 미리 계측하여 메모리의 그 테스트 사이클과 대응한 어드레스에 기억하여 두는 단계,상기 위상차를 구하는 것을 행하는 때에, 각 테스트 사이클마다, 

In [8]:
def build_inputs(ex):
    inputs = dict()
    fields = ["invention_title", "ipc_main", "abstract", "claims"]
    text = " ".join(str(ex[field]) for field in fields if ex[field])   # 빈 필드 skip, 개행/들여쓰기 없음
    return {
        "document_id": ex["document_id"],
        "input" : text,
        "label_ids": ex["label_ids"],
        "kobert_len": ex["kobert_len"],
        "length_bin": ex["length_bin"],
    }

In [ ]:
dataset_inputs = dataset.map(build_inputs, remove_columns=dataset["train"].column_names)
type(dataset_inputs)

In [10]:
dataset_inputs

DatasetDict({
    train: Dataset({
        features: ['document_id', 'label_ids', 'kobert_len', 'length_bin', 'input'],
        num_rows: 201895
    })
    test: Dataset({
        features: ['document_id', 'label_ids', 'kobert_len', 'length_bin', 'input'],
        num_rows: 11271
    })
    val: Dataset({
        features: ['document_id', 'label_ids', 'kobert_len', 'length_bin', 'input'],
        num_rows: 11162
    })
})

In [11]:
dataset_inputs["train"]["input"][0]

'반도체 디바이스 시험방법 및 그의 장치 G01R-031/26 피시험 반도체 디바이스로부터 판독되는 각 데이터와, 이들의 데이터에 동기하여 출력되는 기준 클록을 각각 약간씩 위상차가 부여된 다상 펄스의 스트로브 펄스로 샘플링하고, 이들 샘플링 출력으로부터 각 출력데이터의 변화점 위상과 기준 클록의 변화점 위상을 구하고, 이들 양자의 위상차를 각각 계측하고, 이 위상차가 미리 정한 조건의 범위내에 있는가 여부에 의하여 피시험 반도체 디바이스의 양부를 판정한다. 피시험 디바이스로부터 출력되는 기준클록의 변화점의 초기 위상위치로부터의 위상을 측정하고, 이 기준클록의 위상으로부터 이 기준클록에 동기하여 출력되는 각 데이터의 변화점까지의 위상차를 구하여, 이 위상차의 장단에 의하여 상기 피시험 디바이스의 그레이드를 평가하는 반도체 디바이스 시험방법에 있어서,각 테스트 사이클마다 초기 위상위치로부터 순차적으로 약간씩 위상차가 부여된 다상펄스를 발생시키는 단계,이 다상펄스를 스트로브펄스로 하여 각 테스트 사이클마다 상기 기준클록을 다상으로 샘플링하는 단계,이들 다상 샘플링 출력의 인접하는 출력의 비교로부터 상기 기준클록의 변화점을 검출하고, 이러한 변화점을 검출한 다상펄스의 상번호로부터 상기 기준클록의 변화점의 위상을 결정하는 단계를 포함하는 것을 특징으로 하는 반도체 디바이스 시험방법.피시험 디바이스로부터 출력되는 기준클록의 변화점의 초기 위상위치로부터의 위상을 측정하고, 이 기준클록의 위상으로부터 이 기준클록에 동기하여 출력되는 각 데이터의 변화점까지의 위상차를 구하고, 이 위상차의 장단에 의하여 상기 피시험 디바이스의 그레이드를 평가하는 반도체 디바이스 시험방법에 있어서,각 테스트 사이클마다, 출력되는 상기 기준클록의 변화점의 위상을 미리 계측하여 메모리의 그 테스트 사이클과 대응한 어드레스에 기억하여 두는 단계,상기 위상차를 구하는 것을 행하는 때에, 각 테스트 사이클마다, 상기 메모리의 그 테스트 사이클과 대응한 어드레스로부터 위상을 판독하여, 상기 평가를 행하기

In [25]:
def to_features(ex):
    out = tokenizer(ex["input"])
    batch_size = len(ex["input"])
    y = np.zeros((batch_size, config["num_labels"]), dtype=np.float32)
    for i, ids in enumerate(ex["label_ids"]):
        y[i, ids] = 1.0
    out["labels"] = y.tolist()
    out["kobert_len"] = ex["kobert_len"]
    out["length_bin"] = ex["length_bin"]
    return out

In [ ]:
remove_cols = [c for c in dataset_inputs["train"].column_names if c not in ["document_id", "kobert_len", "length_bin"]]

ds_tok = dataset_inputs.map(
    to_features, 
    remove_columns=remove_cols,
    batched=True
    )

In [29]:
ds_tok

DatasetDict({
    train: Dataset({
        features: ['document_id', 'kobert_len', 'length_bin', 'input_ids', 'attention_mask', 'labels'],
        num_rows: 201895
    })
    test: Dataset({
        features: ['document_id', 'kobert_len', 'length_bin', 'input_ids', 'attention_mask', 'labels'],
        num_rows: 11271
    })
    val: Dataset({
        features: ['document_id', 'kobert_len', 'length_bin', 'input_ids', 'attention_mask', 'labels'],
        num_rows: 11162
    })
})

In [33]:
len(ds_tok["train"][0]["input_ids"]), ds_tok["train"][0]["kobert_len"]

(801, 856)

In [34]:
ds_tok.push_to_hub("ingyoun/patent-clean-text-modernbert-tokenized")

Uploading the dataset shards:   0%|          | 0/3 [00:00<?, ? shards/s]

Creating parquet from Arrow format:   0%|          | 0/4 [00:00<?, ?ba/s]

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

Creating parquet from Arrow format:   0%|          | 0/4 [00:00<?, ?ba/s]

...

Uploading the dataset shards:   0%|          | 0/1 [00:00<?, ? shards/s]

Creating parquet from Arrow format:   0%|          | 0/1 [00:00<?, ?ba/s]

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

CommitInfo(commit_url='https://huggingface.co/datasets/ingyoun/patent-clean-text-modernbert-tokenized/commit/00ca86fe3819f7b37b9d77972ac0b5957e80d135', commit_message='Upload dataset', commit_description='', oid='00ca86fe3819f7b37b9d77972ac0b5957e80d135', pr_url=None, repo_url=RepoUrl('https://huggingface.co/datasets/ingyoun/patent-clean-text-modernbert-tokenized', endpoint='https://huggingface.co', repo_type='dataset', repo_id='ingyoun/patent-clean-text-modernbert-tokenized'), pr_revision=None, pr_num=None)

In [39]:
ds_tok

DatasetDict({
    train: Dataset({
        features: ['document_id', 'kobert_len', 'length_bin', 'input_ids', 'attention_mask', 'labels'],
        num_rows: 201895
    })
    test: Dataset({
        features: ['document_id', 'kobert_len', 'length_bin', 'input_ids', 'attention_mask', 'labels'],
        num_rows: 11271
    })
    val: Dataset({
        features: ['document_id', 'kobert_len', 'length_bin', 'input_ids', 'attention_mask', 'labels'],
        num_rows: 11162
    })
})

In [43]:
def add_length(batch):
    batch["length"] = [len(ids) for ids in batch["input_ids"]]
    batch["diff"] = [
        kb - ln for kb, ln in zip(batch["kobert_len"], batch["length"])
    ]
    return batch

In [44]:
ds_tok = ds_tok.map(add_length, batched=True, num_proc=4)
ds_tok

Map (num_proc=4):   0%|          | 0/201895 [00:00<?, ? examples/s]

Map (num_proc=4):   0%|          | 0/11271 [00:00<?, ? examples/s]

Map (num_proc=4):   0%|          | 0/11162 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['document_id', 'kobert_len', 'length_bin', 'input_ids', 'attention_mask', 'labels', 'length', 'diff'],
        num_rows: 201895
    })
    test: Dataset({
        features: ['document_id', 'kobert_len', 'length_bin', 'input_ids', 'attention_mask', 'labels', 'length', 'diff'],
        num_rows: 11271
    })
    val: Dataset({
        features: ['document_id', 'kobert_len', 'length_bin', 'input_ids', 'attention_mask', 'labels', 'length', 'diff'],
        num_rows: 11162
    })
})

In [45]:
train_lengths = np.array(ds_tok["train"]["length"])
val_lengths = np.array(ds_tok["val"]["length"])
test_lengths = np.array(ds_tok["test"]["length"])

print(f"train length mean={train_lengths.mean():.1f}, median={np.median(train_lengths):.0f}")
print(f"val length mean={val_lengths.mean():.1f}, median={np.median(val_lengths):.0f}")
print(f"test length mean={test_lengths.mean():.1f}, median={np.median(test_lengths):.0f}")

train length mean=795.4, median=628
val length mean=794.8, median=630
test length mean=804.2, median=634


> REF) KoBERT Tokenizer
- train length mean=883.0, median=696
- val length mean=881.4, median=698
- test length mean=892.5, median=703

In [46]:
# kobert와 modernbert 토큰 길이 비교 => kb - mb
train_diff = np.array(ds_tok["train"]["diff"])
val_diff = np.array(ds_tok["val"]["diff"])
test_diff = np.array(ds_tok["test"]["diff"])

print(f"train diff mean={train_diff.mean():.1f}, median={np.median(train_diff):.0f}")
print(f"val diff mean={val_diff.mean():.1f}, median={np.median(val_diff):.0f}")
print(f"test diff mean={test_diff.mean():.1f}, median={np.median(test_diff):.0f}")

train diff mean=87.6, median=63
val diff mean=86.6, median=63
test diff mean=88.4, median=64


In [48]:
def diff_percentile(arr: np.array, metric: str = "None"):
    for p in [50, 75, 90, 95, 99]:
        print(f"p{p} = {np.percentile(arr, p):.0f}")
    if metric == "length":
        print(f"max={arr.max()}, >512 비율={np.mean(arr > 512):.2%}")
    else:
        print(f"max={arr.max()}, min={arr.min()}")


print(f"Train Token Length :")
diff_percentile(train_lengths, "length")
diff_percentile(train_diff)
print(f"\nVal Token Length : ")
diff_percentile(val_lengths, "length")
diff_percentile(val_diff)
print(f"\nTest Token Length : ")
diff_percentile(test_lengths, "length")
diff_percentile(test_diff)

Train Token Length :
p50 = 628
p75 = 928
p90 = 1374
p95 = 1808
p99 = 3621
max=10523, >512 비율=64.54%
p50 = 63
p75 = 112
p90 = 185
p95 = 255
p99 = 503
max=6422, min=-996

Val Token Length : 
p50 = 630
p75 = 923
p90 = 1352
p95 = 1774
p99 = 3769
max=8667, >512 비율=64.76%
p50 = 63
p75 = 111
p90 = 182
p95 = 251
p99 = 495
max=1651, min=-261

Test Token Length : 
p50 = 634
p75 = 936
p90 = 1383
p95 = 1809
p99 = 3750
max=8608, >512 비율=65.15%
p50 = 64
p75 = 113
p90 = 185
p95 = 256
p99 = 517
max=2287, min=-1463
